<div style="background:linear-gradient(90deg,#263238,#455A64);color:#FFFFFF;padding:14px 22px;border-radius:10px;">
<div style="font-size:12px;letter-spacing:2px;text-transform:uppercase;color:#CFD8DC;">Build step 6</div>
<div style="font-size:24px;font-weight:700;">&#128190; Saved queries and visual data prep flow</div>
</div>

<p style="color:#3E2723;font-size:14px;margin:10px 0 4px 0;">The SQL editor script, saved queries 01&ndash;09 and the Loyalty Promo Target List Prep flow. The warehouse, dashboard, Genie agent and alerts are bundle resources and are deployed by the bundle itself.</p>


In [0]:
from bean_and_brew import init, fill


In [0]:
# Saved queries and the visual data prep flow have no bundle resource type, so this task creates them inside the
# bundle's own folder (${workspace.root_path}/assets). `databricks bundle destroy` deletes that folder with everything in it.
# Only objects this task created (their IDs are kept in asset_ids.json in that folder) are ever updated; nothing is
# looked up or adopted by name, so an existing object with the same name is never touched.
import base64, html, json, os, urllib.parse
from databricks.sdk import WorkspaceClient

cfg = init(spark, dbutils, extra=("warehouse_id", "llm_endpoint", "asset_prefix", "assets_folder"))
w = WorkspaceClient()
api = w.api_client.do

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SRC = os.path.normpath(os.path.join(os.path.dirname("/Workspace" + nb_path.removeprefix("/Workspace")), "..", "assets"))
MANIFEST = json.load(open(os.path.join(SRC, "manifest.json")))
PREFIX = cfg["asset_prefix"]
ASSETS_DIR = cfg["assets_folder"].removeprefix("/Workspace")
WS_DIR = "/Workspace" + ASSETS_DIR
STATE_PATH = f"{ASSETS_DIR}/asset_ids.json"

def template(rel):
    with open(os.path.join(SRC, rel)) as f:
        return fill(f.read(), cfg)

def b64(s):
    return base64.b64encode(s.encode("utf-8")).decode()

def get_or_none(path):
    try:
        return api("GET", path)
    except Exception:
        return None

def status(path):
    return get_or_none("/api/2.0/workspace/get-status?" + urllib.parse.urlencode({"path": path}))

api("POST", "/api/2.0/workspace/mkdirs", body={"path": ASSETS_DIR})
try:
    state = json.loads(base64.b64decode(api("GET", "/api/2.0/workspace/export", query={"path": STATE_PATH, "format": "AUTO"})["content"]))
except Exception:
    state = {}

def save_state():  # written after every asset, so a failed run can be re-run without creating duplicates
    api("POST", "/api/2.0/workspace/import", body={"path": STATE_PATH, "format": "AUTO", "overwrite": True, "content": b64(json.dumps(state, indent=2))})

def upsert_query(key, name, text, description=""):
    body = {"display_name": PREFIX + name, "query_text": text, "warehouse_id": cfg["warehouse_id"], "description": description}
    qid = state.get(key)
    current = get_or_none(f"/api/2.0/sql/queries/{qid}") if qid else None
    if current and current.get("lifecycle_state", "ACTIVE") == "ACTIVE":
        api("PATCH", f"/api/2.0/sql/queries/{qid}", body={"update_mask": "display_name,query_text,warehouse_id,description", "query": body})
    else:
        qid = api("POST", "/api/2.0/sql/queries", body={"query": {**body, "parent_path": WS_DIR}})["id"]
    state[key] = qid
    save_state()

def statements(parts):
    return ";\n\n".join(template(f"queries/{p}.sql").strip() for p in parts)

# SQL editor: the 11-statement live script, then one saved query per feature
s = MANIFEST["script"]
upsert_query(s["key"], s["name"], statements(s["parts"]) + ";", s["description"])
for q in MANIFEST["saved_queries"]:
    upsert_query(q["key"], q["name"], statements(q["parts"]))

# Visual data prep flow: the designer file, pointed at the bundle's catalog and schemas
flow = MANIFEST["flow"]
flow_path = f"{ASSETS_DIR}/{PREFIX}{flow['name']}"
exists = status(flow_path) is not None
if exists and state.get("flow") != flow_path:
    raise RuntimeError(f"{flow_path} already exists and was not created by this bundle, so it is left untouched.")
api("POST", "/api/2.0/workspace/import", body={"path": flow_path, "format": "AUTO", "overwrite": exists, "content": b64(template(flow["file"]))})
state["flow"] = flow_path
save_state()
print(f"Saved queries and the data prep flow are in {ASSETS_DIR} (warehouse {cfg['warehouse_id']}, ai_query endpoint {cfg['llm_endpoint']})")


In [0]:
# Asset index with live links for this workspace
def object_id(path):
    s = status(path)
    return s.get("object_id") if s else None

def link(href, text):
    return f'<a href="{href}" target="_blank">{html.escape(text)}</a>' if href else html.escape(text)

C = cfg["catalog"]
rows = [(link(f"/browse/folders/{object_id(ASSETS_DIR)}", ASSETS_DIR.rsplit("/", 1)[-1] + " (bundle folder)"), "Folder", "Saved queries and the data prep flow")]
rows += [(link(f"/explore/data/{C}/{cfg[k]}", f"{C}.{cfg[k]}"), "Schema", p) for k, p in
         [("bronze_schema", "Raw source data, defects included"), ("silver_schema", "Cleaned dimensions and facts with keys and tags"), ("gold_schema", "Business tables and the 3 metric views")]]
rows.append((link(f"/sql/editor/{state[s['key']]}", PREFIX + s["name"]), "Query", "11-statement demo script"))
rows += [(link(f"/sql/editor/{state[q['key']]}", PREFIX + q["name"]), "Query", "Saved query, one feature each") for q in MANIFEST["saved_queries"]]
flow_id = object_id(flow_path)
rows.append((link(f"/editor/designer-files/{flow_id}" if flow_id else None, PREFIX + flow["name"].removesuffix(".designer.ipynb")), "Visual data prep", "No-code promo list for marketing"))
rows.append(("Warehouse, dashboard, Genie agent, 3 alerts, jobs", "Bundle resources", "Listed in the bundle's Deployments panel (open databricks.yml)"))

body = "".join(f'<tr style="background:{"#FBF6EF" if i % 2 else "#FFFFFF"};"><td style="padding:5px 8px;">{a}</td><td style="padding:5px 8px;">{t}</td><td style="padding:5px 8px;">{p}</td></tr>'
               for i, (a, t, p) in enumerate(rows))
displayHTML(f"""<div style="font-family:sans-serif;color:#3E2723;"><div style="font-weight:700;font-size:16px;margin-bottom:6px;">&#128193; Asset index</div>
<table style="border-collapse:collapse;font-size:13px;width:100%;"><tr style="background:#5D4037;color:#FFF8F0;">
<th style="text-align:left;padding:5px 8px;">Asset</th><th style="text-align:left;padding:5px 8px;">Type</th><th style="text-align:left;padding:5px 8px;">Purpose</th></tr>{body}</table></div>""")
